================================================================================================================
The following is to calculate first component of SH for **GP1** using PCA
================================================================================================================
CV will be performed and one fold for PCA, the other four for ML
================================================================================================================

In [ ]:
import sys
sys.path.append('working_path')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.model_selection import KFold

from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
from sklearn.compose import ColumnTransformer
from confound_removal import ConfoundRemover

In [ ]:
df = pd.read_csv('working_path/cross_reversed.csv')

In [ ]:
# Define original variables
Age = ['age']
Sex = ['Sex']
sleep = ['sleep_variables']

confounds = Age # Sex not included for normalization 

confounds_data = df[confounds].values
Sex_data = df[Sex].values

# Stack confounds with sex data
join_confounds = np.column_stack((confounds_data, Sex_data))

In [ ]:
# Define feature group for scaling

features_to_scale = sleep + confounds
features_to_skip = Sex
input_cols = features_to_scale + features_to_skip 

# Setup the preprocessor
preprocessor_sleep = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_to_scale)
        ],
        remainder='passthrough' # N_12 and sex are binary
)

confoundremover_order = sleep + Age + Sex

In [6]:
# Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# cross-validation with 1 fold for PCA and 4 folds for ML
for i, (ML_idx, PCA_idx) in enumerate(kf.split(df)):
    # Split data
    train_PCA_raw = df.iloc[PCA_idx][input_cols] 
    test_ML_raw = df.iloc[ML_idx][input_cols]

    # Scaling
    train_PCA_scaled_arr = preprocessor_sleep.fit_transform(train_PCA_raw)
    test_ML_scaled_arr = preprocessor_sleep.transform(test_ML_raw)

    # Convert to DataFrame
    train_scaled_df = pd.DataFrame(train_PCA_scaled_arr, columns=input_cols)
    test_ML_scaled_df = pd.DataFrame(test_ML_scaled_arr, columns=input_cols)

    # Reorder features for confounds removal
    train_PCA_ordered = train_scaled_df[confoundremover_order].values
    test_ML_ordered = test_ML_scaled_df[confoundremover_order].values

    # Deconfound
    remover = ConfoundRemover(join_confounds.shape[1])
    train_PCA_deconfound = remover.fit_transform(train_PCA_ordered)
    test_ML_deconfound = remover.transform(test_ML_ordered)

    # Run PCA to get eigenvalues
    fa = FactorAnalyzer(rotation='varimax', method='principal')
    fa.fit(train_PCA_deconfound[:, :len(sleep)])
    ev_sleep, v_sleep = fa.get_eigenvalues()

    # Determine number of components with eigenvalue > 1
    n_components_sleep = sum(ev_sleep > 1)
    print(f"Number of components for SH with eigenvalue > 1 in fold {i+1}: {n_components_sleep}")

    # Get rotated loadings for sleep
    loadings_sleep = pd.DataFrame(fa.loadings_, index=df[sleep].columns)
    print(f"Rotated component loadings for sleep in fold {i+1}:\n", loadings_sleep)

    # Get variance explained by each component for sleep
    variance_sleep = fa.get_factor_variance()
    explained_variance_sleep_df = pd.DataFrame({
        'SS Loadings': variance_sleep[0],
        'Proportion Var': variance_sleep[1],
        'Cumulative Var': variance_sleep[2]
    }, index=[f'PC{i+1}' for i in range(len(variance_sleep[0]))])
    print("\nExplained Variance:\n", explained_variance_sleep_df)

    # Transform to ML data
    sleep_scores = fa.transform(test_ML_deconfound[:, :len(sleep)])
    
    # Plot the loadings for sleep
    plt.figure(figsize=(10, 6))
    sns.heatmap(loadings_sleep, annot=True, cmap='viridis', cbar=True, center=0)
    plt.title(f'Rotated Component Loadings for sleep in fold {i+1}')
    plt.xlabel('Components')
    plt.ylabel('Sleep')
    plt.tight_layout()
    plt.show()

    # Select rows that match the ML_idx
    df.loc[df.index[ML_idx], 'pca_comp1_SH'] = sleep_scores[:, 0]
    df_results = df.iloc[ML_idx]

    # Save files
    df_results.to_csv(f'working_path/PCA_PC1_SH_fold_{i+1}.csv', index=False)
        